In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

import joblib
import os

import math

In [ ]:
file_path = "data/italian seriaA data.csv"

df = pd.read_csv(file_path)

# Mostrare le prime 5 righe
print("Prime 5 righe del dataset")
display(df.head())

# Mostrare le informazioni fondamentali
print("\nInformazioni sulle colonne")
df.info()

In [ ]:
# Rimuoviamo gli spazi vuoti accidentali all'inizio o alla fine dei nomi delle colonne
df.columns = df.columns.str.strip()

# Stampiamo la lista esatta delle colonne
print("Ecco la lista esatta delle tue colonne pulite:")
print(df.columns.tolist())

In [ ]:
# STEP 1 : creazione del TARGET
# impostato a PAREGGIO di default
df['Target'] = 0

# se i gol in casa superano quelli in trasferta allora VITTORIA CASA
df.loc[df['Home Team Goals Scored'] > df['Away Team Goals Scored'], 'Target'] = 1

# se i gol in trasferta superano quelli in casa allora VITTORIA TRASFERTA
df.loc[df['Home Team Goals Scored'] < df['Away Team Goals Scored'], 'Target'] = 2

df['Home Team xG'] = (df['Home Team On Target Shots'] * 0.25) + (df['Home Team Off Target Shots'] * 0.05)
df['Away Team xG'] = (df['Away Team On Target Shots'] * 0.25) + (df['Away Team Off Target Shots'] * 0.05)

# STEP 2 : Calcolo STATO DI FORMA
def calcola_statistiche_recenti(df, nome_squadra, indice_corrente, n_partite = 3):
    # si considerano tutte le righe prima di quella corrente
    partite_passate = df.iloc[:indice_corrente]

    # si filtrano solo le partite in cui ha giocato la nostra squadra (casa o trasferta)
    partite_squadra = partite_passate[(partite_passate['Home Team'] == nome_squadra)
                                      | (partite_passate['Away Team'] == nome_squadra)
                                      ].tail(n_partite)
    
    if len(partite_squadra) < n_partite:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    
    gol_fatti = 0
    gol_subiti = 0
    tiri_fatti = 0
    tiri_subiti = 0
    possesso = 0
    angoli_a_favore = 0
    angoli_contro = 0
    xg_squadra = 0

    # si sommano i gol
    for idx, row in partite_squadra.iterrows():
        if row['Home Team'] == nome_squadra:
            gol_fatti += row['Home Team Goals Scored']
            gol_subiti += row['Away Team Goals Scored']
            tiri_fatti += row['Home Team On Target Shots']
            tiri_subiti += row['Away Team On Target Shots']
            possesso += row['Home Team Possession %']
            angoli_a_favore += row['Home Team Corners']
            angoli_contro += row['Away Team Corners']
            xg_squadra += row['Home Team xG']
        else:
            gol_fatti += row['Away Team Goals Scored']
            gol_subiti += row['Home Team Goals Scored']
            tiri_fatti += row['Away Team On Target Shots']
            tiri_subiti += row['Home Team On Target Shots']
            possesso += row['Away Team Possession %']
            angoli_a_favore += row['Away Team Corners']
            angoli_contro += row['Home Team Corners']
            xg_squadra += row['Away Team xG']

    return (gol_fatti / n_partite , gol_subiti / n_partite,
            tiri_fatti / n_partite, tiri_subiti / n_partite,
            possesso / n_partite,
            angoli_a_favore / n_partite, angoli_contro / n_partite,
            xg_squadra / n_partite)

colonne_statistiche = [
'Home_Form_Gol_Fatti','Home_Form_Gol_Subiti','Away_Form_Gol_Fatti','Away_Form_Gol_Subiti',
'Home_Form_Tiri_Fatti','Home_Form_Tiri_Subiti','Away_Form_Tiri_Fatti','Away_Form_Tiri_Subiti',
'Home_Form_Possesso','Away_Form_Possesso',
'Home_Form_Angoli_Fatti','Home_Form_Angoli_Subiti','Away_Form_Angoli_Fatti','Away_Form_Angoli_Subiti',
'Home_Form_xG', 'Away_Form_xG']

for col in colonne_statistiche:
    df[col] = 0.0

for idx, row in df.iterrows():
    # Calcolo per la squadra in casa
    (hf_gf, hf_gs, hf_tf, hf_ts, hf_p, hf_af, hf_as, hf_xg) = calcola_statistiche_recenti(df, row['Home Team'], idx)
    df.at[idx, 'Home_Form_Gol_Fatti'] = hf_gf
    df.at[idx, 'Home_Form_Gol_Subiti'] = hf_gs
    df.at[idx, 'Home_Form_Tiri_Fatti'] = hf_tf
    df.at[idx, 'Home_Form_Tiri_Subiti'] = hf_ts
    df.at[idx, 'Home_Form_Possesso'] = hf_p
    df.at[idx, 'Home_Form_Angoli_Fatti'] = hf_af
    df.at[idx, 'Home_Form_Angoli_Subiti'] = hf_as
    df.at[idx, 'Home_Form_xG'] = hf_xg
    
    # Calcolo per la squadra in trasferta
    (af_gf, af_gs, af_tf, af_ts, af_p, af_af, af_as, af_xg) = calcola_statistiche_recenti(df, row['Away Team'], idx)
    df.at[idx, 'Away_Form_Gol_Fatti'] = af_gf
    df.at[idx, 'Away_Form_Gol_Subiti'] = af_gs
    df.at[idx, 'Away_Form_Tiri_Fatti'] = af_tf
    df.at[idx, 'Away_Form_Tiri_Subiti'] = af_ts
    df.at[idx, 'Away_Form_Possesso'] = af_p
    df.at[idx, 'Away_Form_Angoli_Fatti'] = af_af
    df.at[idx, 'Away_Form_Angoli_Subiti'] = af_as
    df.at[idx, 'Away_Form_xG'] = af_xg

print("Calcolo terminato!")


1. La Scelta del Modello

In [ ]:
# Si definiscono X --> le variabili da studiare e y --> il target
X = df[colonne_statistiche]
y = df['Target']

# SPLIT (80% Train, 20% Test)
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]

X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print(f"Partite usate per imparare (Train): {len(X_train)}")
print(f"Partite usate per l'esame (Test): {len(X_test)}")

# CREAZIONE ED ADDESTRAMENTO DEL MODELLO
modello = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
modello.fit(X_train, y_train)

# Previsioni 
y_pred = modello.predict(X_test)

# Valutazione
accuratezza = accuracy_score(y_test, y_pred)
print(f"L'accuratezza del modello base è: {accuratezza * 100: .2f}%")

# Confronto delle prime 10 previsioni con la realtà
confronto = pd.DataFrame({
    'Partita': df.iloc[split_index:split_index+10]['Home Team'] + ' vs '
    + df.iloc[split_index:split_index+10]['Away Team'],
    'Previsione del modello': y_pred[:10],
    'Risultato Reale': y_test[:10].values
})
display(confronto)

In [ ]:
# si estrae l'importanza delle feature direttamente dal modello
importanza = modello.feature_importances_

# creazione della tabella per la visualizzazione
df_importanza = pd.DataFrame({
    'Statistica': colonne_statistiche,
    'Importanza': importanza
})

# ordinamento dalla meno importante alla più importante
df_importanza = df_importanza.sort_values(by='Importanza', ascending=True)

# grafico
plt.figure(figsize=(10, 8))
plt.barh(df_importanza['Statistica'], df_importanza['Importanza'], color='teal')
plt.xlabel('Peso decisionale dell\'algoritmo')
plt.title('Quali statistiche guarda di più il modello per indovinare il risultato?')
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
# Calcolo della matrice di confusione
matrice_confusione = confusion_matrix(y_test, y_pred)

# Creazione della HEATMAP
plt.figure(figsize=(8, 6))
sns.heatmap(matrice_confusione, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Previsto: Pareggio (0)', 'Previsto: Casa (1)', 'Previsto: Trasferta (2)'],
            yticklabels=['Reale: Pareggio (0)', 'Reale: Casa (1)', 'Reale: Trasferta (2)'])

plt.title('Matrice di Confusione')
plt.xlabel('Previsioni del Modello')
plt.ylabel('Risultati Reali')
plt.show()

In [ ]:
# Calcolo dei TESTA A TESTA
def calcola_h2h(df, squadra_casa, squadra_trasferta, indice_corrente, n_scontri = 5):
    # si considera tutto il passato
    storico = df.iloc[:indice_corrente]

    # si filtrano SOLO le partite dove queste squadre si sono incontrate
    scontri_diretti = storico[
        ((storico['Home Team'] == squadra_casa) & (storico['Away Team'] == squadra_trasferta)) 
        | ((storico['Home Team'] == squadra_trasferta) & (storico['Away Team'] == squadra_casa)) 
    ].tail(n_scontri)

    vittorie_casa_h2h = 0
    vittorie_trasferta_h2h = 0
    pareggi_h2h = 0

    if len(scontri_diretti) == 0:
        return 0.0, 0.0, 0.0
    
    for idx, row in scontri_diretti.iterrows():
        # si allineano i risultati storici a chi gioca in casa oggi
        if row['Home Team'] == squadra_casa:
            if row['Target'] == 1: vittorie_casa_h2h += 1
            elif row['Target'] == 2: vittorie_trasferta_h2h += 1
            else: pareggi_h2h += 1
        else:
            # la squadra che gioca oggi in casa, nello scontro storico era in trasferta
            if row['Target'] == 2: vittorie_casa_h2h += 1
            elif row['Target'] == 1: vittorie_trasferta_h2h += 1
            else: pareggi_h2h += 1
    
    totale = len(scontri_diretti)

    return vittorie_casa_h2h / totale, vittorie_trasferta_h2h / totale,  pareggi_h2h / totale

df['H2H_Vittorie_Casa'] = 0.0
df['H2H_Pareggi'] = 0.0
df['H2H_Vittorie_Trasferta'] = 0.0

for idx, row in df.iterrows():
    v_casa, pareggi, v_trasf = calcola_h2h(df, row['Home Team'], row['Away Team'], idx)
    df.at[idx, 'H2H_Vittorie_Casa'] = v_casa
    df.at[idx, 'H2H_Pareggi'] = pareggi
    df.at[idx, 'H2H_Vittorie_Trasferta'] = v_trasf

print("Testa a Testa calcolati!")
display(df[['Home Team', 'Away Team', 'H2H_Vittorie_Casa', 'H2H_Pareggi', 'H2H_Vittorie_Trasferta']].tail(5))

In [ ]:
# calcolo della distanza percorsa dalla squadra in trasferta

# 1. Dizionario geografico: Latitudine e Longitudine delle città di Serie A
coordinate_citta = {
    'JUVENTUS': (45.0703, 7.6868), 'TORINO': (45.0703, 7.6868),          # Torino
    'MILAN': (45.4642, 9.1900),  'INTER': (45.4642, 9.1900),             # Milano
    'ROMA': (41.9028, 12.4964), 'LAZIO': (41.9028, 12.4964),             # Roma
    'NAPOLI': (40.8518, 14.2681),                                        # Napoli
    'ATALANTA': (45.6983, 9.6773),                                       # Bergamo
    'FIORENTINA': (43.7696, 11.2558),                                    # Firenze
    'BOLOGNA': (44.4949, 11.3426),                                       # Bologna
    'SASSUOLO': (44.5434, 10.9135),                                      # Reggio Emilia
    'HELLAS': (45.4384, 10.9916), 'CHIEVO': (45.4384, 10.9916),                          # Verona
    'GENOA': (44.4056, 8.9463), 'SAMPDORIA': (44.4056, 8.9463),          # Genova
    'LECCE': (40.3515, 18.1750),                                         # Lecce
    'SALERNITANA': (40.6824, 14.7681),                                   # Salerno
    'UDINESE': (46.0619, 13.2378),                                       # Udine
    'CAGLIARI': (39.2238, 9.1116),                                       # Cagliari
    'EMPOLI': (43.7181, 10.9453),                                        # Empoli
    'MONZA': (45.5845, 9.2735),                                          # Monza
    'FROSINONE': (41.6398, 13.3414),                                     # Frosinone
    'SPEZIA': (44.1025, 9.8241),                                         # La Spezia
    'VENEZIA': (45.4408, 12.3155),                                       # Venezia
    'CREMONESE': (45.1332, 10.0256),                                     # Cremona
    'PARMA': (44.8015, 10.3279),                                         # Parma
    'BRESCIA': (45.5416, 10.2168),                                       # Brescia
    'SPAL': (44.8381, 11.6198),                                          # Ferrara
    'BENEVENTO': (41.1329, 14.7735),                                     # Benevento
    'CROTONE': (39.0808, 17.1276),                                       # Crotone
    'CESENA': (44.1391, 12.2432),                                         # Cesena
    'CARPI': (44.7833, 10.8833),                                         # Carpi
    'PESCARA': (42.4618, 14.2161),                                       # Pescara
    'PALERMO': (38.1157, 13.3615)                                        # Palermo
}

# 2. Funzione matematica per calcolare la distanza in KM sulla sfera terrestre
def calcola_distanza_km(lat1, lon1, lat2, lon2):
    R = 6371.0 # Raggio della Terra in km
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# 3. Creiamo la nuova feature nel dataset
df['Fatica_Trasferta_km'] = 0.0

squadre_non_trovate = set()

for idx, row in df.iterrows():
    sq_casa = row['Home Team'].strip()
    sq_trasferta = row['Away Team'].strip()
    
    # Verifichiamo di avere le coordinate di entrambe
    if sq_casa in coordinate_citta and sq_trasferta in coordinate_citta:
        coord_casa = coordinate_citta[sq_casa]
        coord_trasferta = coordinate_citta[sq_trasferta]
        
        # Calcoliamo i KM che la squadra in trasferta ha dovuto viaggiare per arrivare allo stadio
        distanza = calcola_distanza_km(coord_casa[0], coord_casa[1], coord_trasferta[0], coord_trasferta[1])
        df.at[idx, 'Fatica_Trasferta_km'] = distanza
    else:
        # Se manca qualche squadra nel dizionario, la salviamo per aggiungerla
        if sq_casa not in coordinate_citta: squadre_non_trovate.add(sq_casa)
        if sq_trasferta not in coordinate_citta: squadre_non_trovate.add(sq_trasferta)

if squadre_non_trovate:
    print(f"🚨 ATTENZIONE! Manca la città per queste squadre: {squadre_non_trovate}")
    print("Aggiungile al dizionario 'coordinate_citta' per non avere chilometri a zero!")
else:
    print("✅ Calcolo completato con successo per tutte le partite!")


# Mostriamo un assaggio dei risultati
# display(df[['Home Team', 'Away Team', 'Fatica_Trasferta_km']].tail(10))

In [ ]:
# Aggiungiamo le nuove colonne dei TESTA A TESTA alla lista di feature
colonne_finali = colonne_statistiche + ['H2H_Vittorie_Casa', 'H2H_Pareggi', 'H2H_Vittorie_Trasferta', 
                                        'Fatica_Trasferta_km']

X = df[colonne_finali]
y = df['Target']

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

modello_definitivo = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
modello_definitivo.fit(X_train, y_train)

previsioni_finali = modello_definitivo.predict(X_test)
accuratezza_finale = accuracy_score(y_test, previsioni_finali)

print(f"\nL'accuratezza del MODELLO DEFINITIVO è: {accuratezza_finale * 100:.2f}%")

In [ ]:
# Creiamo una cartella 'modelli' se non esiste già
os.makedirs('modelli', exist_ok=True)

# 1. Salviamo il modello addestrato
percorso_modello = 'modelli/random_forest_calcio.pkl'
joblib.dump(modello_definitivo, percorso_modello)

# 2. Salviamo anche la lista delle colonne, così in futuro sapremo esattamente 
# quali dati dare in pasto al modello per fare una previsione
percorso_colonne = 'modelli/colonne_modello.pkl'
joblib.dump(colonne_finali, percorso_colonne)

print(f"Modello salvato con successo in: {percorso_modello}")
print("Ora puoi chiudere il computer, riaprirlo domani, e usare il modello senza doverlo riaddestrare!")

SIMULAZIONE


In [ ]:
# 1. Carichiamo il "cervello" del nostro modello salvato
modello = joblib.load('modelli/random_forest_calcio.pkl')
colonne = joblib.load('modelli/colonne_modello.pkl')

# 2. Creiamo la nostra partita immaginaria inserendo le statistiche
partita_live = {
    # Statistiche Squadra in Casa (Molto in forma)
    'Home_Form_Gol_Fatti': [2.0],
    'Home_Form_Gol_Subiti': [0.5],
    'Home_Form_Tiri_Fatti': [6.0],
    'Home_Form_Tiri_Subiti': [2.0],
    'Home_Form_Possesso': [60.0],
    'Home_Form_Angoli_Fatti': [7.0],
    'Home_Form_Angoli_Subiti': [3.0],
    
    # Statistiche Squadra in Trasferta (Fuori forma)
    'Away_Form_Gol_Fatti': [0.5],
    'Away_Form_Gol_Subiti': [1.5],
    'Away_Form_Tiri_Fatti': [2.0],
    'Away_Form_Tiri_Subiti': [5.0],
    'Away_Form_Possesso': [40.0],
    'Away_Form_Angoli_Fatti': [3.0],
    'Away_Form_Angoli_Subiti': [6.0],
    
    # Testa a Testa Storici (La "Bestia Nera": vince quasi sempre la squadra in trasferta)
    'H2H_Vittorie_Casa': [0.0],
    'H2H_Pareggi': [0.2],
    'H2H_Vittorie_Trasferta': [0.8]
}

# Trasformiamo i dati in una tabella leggibile dal modello
df_live = pd.DataFrame(partita_live, columns=colonne)

# 3. Chiediamo al modello la Previsione e le Probabilità
previsione = modello.predict(df_live)[0]
probabilita = modello.predict_proba(df_live)[0] # Estraiamo le percentuali (1, X, 2)

# Mappiamo il risultato numerico a qualcosa di leggibile
risultato_testo = {0: "Pareggio (X)", 1: "Vittoria Casa (1)", 2: "Vittoria Trasferta (2)"}

print("--- PREVISIONE DELLA PARTITA ---")
print(f"Esito consigliato: {risultato_testo[previsione]}\n")

print("Dettaglio Probabilità calcolate:")
print(f"- Pareggio: {probabilita[0] * 100:.1f}%")
print(f"- Vittoria Casa: {probabilita[1] * 100:.1f}%")
print(f"- Vittoria Trasferta: {probabilita[2] * 100:.1f}%")

----------------------------------------------------------------------------------

PREDIZIONE PARTITE FUTURE

In [ ]:
# lista esatta delle squadre nel dataset
print(df['Home Team'].unique())

In [ ]:
def predici_partita(squadra_casa, squadra_trasferta, assenze_casa=None, assenze_trasferta=None):
    if assenze_casa is None: assenze_casa = []
    if assenze_trasferta is None: assenze_trasferta = []

    indice_attuale = len(df)
    
    # 1. Calcolo Statistiche di Base e H2H
    (hf_gf, hf_gs, hf_tf, hf_ts, hf_p, hf_af, hf_as, hf_xg) = calcola_statistiche_recenti(df, squadra_casa, indice_attuale)
    (af_gf, af_gs, af_tf, af_ts, af_p, af_af, af_as, af_xg) = calcola_statistiche_recenti(df, squadra_trasferta, indice_attuale)
    v_casa_h2h, v_trasf_h2h, pareggi_h2h = calcola_h2h(df, squadra_casa, squadra_trasferta, indice_attuale)
    
    # 2. CALCOLO FATICA DA TRASFERTA
    km_trasferta = 0.0
    sq_casa_upper = squadra_casa.upper()
    sq_trasf_upper = squadra_trasferta.upper()
    
    if sq_casa_upper in coordinate_citta and sq_trasf_upper in coordinate_citta:
        coord_c = coordinate_citta[sq_casa_upper]
        coord_t = coordinate_citta[sq_trasf_upper]
        km_trasferta = calcola_distanza_km(coord_c[0], coord_c[1], coord_t[0], coord_t[1])
    else:
        print(f"⚠️ Attenzione: Coordinate non trovate. La distanza sarà 0 km.")


    malus_casa = []
    malus_trasferta = []

    # Analisi assenze per la squadra in casa
    for ruolo in assenze_casa:
        ruolo = ruolo.lower()
        if ruolo == 'attaccante':
            hf_xg *= 0.75 # perde il 25% di xG
            hf_tf *= 0.80 # perde il 20% di tiri
            malus_casa.append("-25% Potenziale offensivo")
        elif ruolo == 'centrocampista':
            hf_p *= 0.90 # perde il 10% di possesso
            hf_tf *= 0.85 # perde il 15% di xG
            malus_casa.append("-10% Possesso, -15 XG")
        elif ruolo in ['difensore','portiere']:
            af_xg *= 1.25
            af_tf *= 1.20
            malus_casa.append("Difesa fragile")
    
    # Analisi assenze per la squadra in trasferta
    for ruolo in assenze_trasferta:
        ruolo = ruolo.lower()
        if ruolo == 'attaccante':
            af_xg *= 0.75 # perde il 25% di xG
            af_tf *= 0.80 # perde il 20% di tiri
            malus_casa.append("-25% Potenziale offensivo")
        elif ruolo == 'centrocampista':
            af_p *= 0.90 # perde il 10% di possesso
            af_tf *= 0.85 # perde il 15% di xG
            malus_casa.append("-10% Possesso, -15 XG")
        elif ruolo in ['difensore','portiere']:
            hf_xg *= 1.25
            hf_tf *= 1.20
            malus_casa.append("Difesa fragile")

    # 3. Creazione del pacchetto dati
    dati_partita = {
        'Home_Form_Gol_Fatti': [hf_gf], 'Home_Form_Gol_Subiti': [hf_gs],
        'Away_Form_Gol_Fatti': [af_gf], 'Away_Form_Gol_Subiti': [af_gs],
        'Home_Form_Tiri_Fatti': [hf_tf], 'Home_Form_Tiri_Subiti': [hf_ts],
        'Away_Form_Tiri_Fatti': [af_tf], 'Away_Form_Tiri_Subiti': [af_ts],
        'Home_Form_Possesso': [hf_p], 'Away_Form_Possesso': [af_p],
        'Home_Form_Angoli_Fatti': [hf_af], 'Home_Form_Angoli_Subiti': [hf_as],
        'Away_Form_Angoli_Fatti': [af_af], 'Away_Form_Angoli_Subiti': [af_as],
        'H2H_Vittorie_Casa': [v_casa_h2h], 
        'H2H_Pareggi': [v_trasf_h2h], 
        'H2H_Vittorie_Trasferta': [pareggi_h2h],
        'Fatica_Trasferta_km': [km_trasferta],
        'Home_Form_xG': [hf_xg], 'Away_Form_xG': [af_xg]
    }
    
    df_live = pd.DataFrame(dati_partita, columns=colonne_finali) # Usiamo colonne_finali aggiornate
    
    if (df_live['Home_Form_Possesso'][0] == 0.0) or (df_live['Away_Form_Possesso'][0] == 0.0):
        print("🚨 ATTENZIONE: Il modello vede tutti ZERI! Controlla i nomi inseriti.")
        return 
        
    print("="*45)
    print(f"🔥 CONDIZIONI DEL MATCH, CHIMICA e ASSENZE")
    print("="*45)
    print(f"✈️ Fatica in viaggio per {squadra_trasferta}: {km_trasferta:.0f} km")
    if assenze_casa or assenze_trasferta:
        print("\n🚑 REPORT INDISPONIBILI:")
        if assenze_casa:
            print(f"   ❌ {squadra_casa.upper()} perde: {', '.join(assenze_casa)}")
            for malus in malus_casa: print(f"        ↳ {malus}")
        if assenze_trasferta:
            print(f"   ❌ {squadra_trasferta.upper()} perde: {', '.join(assenze_trasferta)}")
            for malus in malus_trasferta: print(f"       ↳ {malus}")

    print(f"\n🏠 {squadra_casa.upper()}:")
    print(f"   xG POTENZIALI: {hf_xg:.1f} | Tiri previsti: {hf_tf:.1f} | Possesso palla: {hf_p:.1f}%")
    print(f"✈️ {squadra_trasferta.upper()}:")
    print(f"   xG POTENZIALI: {af_xg:.1f} | Tiri previsti: {af_tf:.1f} | Possesso palla: {af_p:.1f}%")
    
    # 4. STORICO TESTA A TESTA
    print("\n" + "="*45)
    print(f"🔄 STORICO ULTIMI SCONTRI DIRETTI")
    print("="*45)
    scontri_diretti = df[
        ((df['Home Team'] == squadra_casa) & (df['Away Team'] == squadra_trasferta)) |
        ((df['Home Team'] == squadra_trasferta) & (df['Away Team'] == squadra_casa))
    ].tail(5)
    
    if len(scontri_diretti) == 0:
        print("Nessun precedente storico trovato nel dataset.")
    else:
        for idx, row in scontri_diretti.iterrows():
            print(f"📅 Stagione {int(row['year'])} | {row['Home Team']} {int(row['Home Team Goals Scored'])} - {int(row['Away Team Goals Scored'])} {row['Away Team']}")
    
    # 5. Previsione 
    previsione = modello_definitivo.predict(df_live)[0]
    probabilita = modello_definitivo.predict_proba(df_live)[0]
    risultato_testo = {0: "Pareggio (X)", 1: "Vittoria Casa (1)", 2: "Vittoria Trasferta (2)"}
    
    print("\n" + "="*45)
    print(f"🤖 ESITO CONSIGLIATO: {risultato_testo[previsione]}")
    print("="*45)
    print(f"1 (Vittoria {squadra_casa}): {probabilita[1] * 100:.1f}%")
    print(f"X (Pareggio): {probabilita[0] * 100:.1f}%")
    print(f"2 (Vittoria {squadra_trasferta}): {probabilita[2] * 100:.1f}%")

In [ ]:
predici_partita('JUVENTUS','ROMA', assenze_casa=[], assenze_trasferta=['attaccante','difensore'])